<a href="https://colab.research.google.com/github/packtpublishing/Machine-Learning-for-Imbalanced-Data/blob/main/chapter10/model_calibration_on_Kaggle_HR_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### [Brief about Data](https://www.kaggle.com/datasets/jacksonchou/hr-data-for-analytics#Breif-about-Data-:)

Data is about Human resources in a specific company each record represents an employee

#### [Variables description](https://www.kaggle.com/code/ahmedhas93/hr-data-cleaning-outliers-analysis-eda-lr-rf/notebook#Variables-description-:)

Dependent variable

-   Left : 0 if employee did not leave , 1 if left company

Independent variables

-   satisfaction_level : means how much employee satisfied (0 less satisfied , 1 most satisfied)
-   last_evaluation : means employees' evaluation for last month (0 bad , 1 Excellent)
-   number_project : number of projects the employee worked on
-   average_montly_hours : average months employee spends at work per month
-   time_spend_company : years the employee spent in a company
-   Work_accident : 0 if he did not have an accident , 1 if had at least one
-   promotion_last_5years : 0 if he did not have any promotion in last 5 years , 1 if had at least one
-   sales : department in which employee works


# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import brier_score_loss

In [ ]:
!wget https://raw.githubusercontent.com/PacktPublishing/Machine-Learning-for-Imbalanced-Data/main/chapter10/HR_comma_sep.csv

# EDA

In [ ]:
# Load the dataset
df = pd.read_csv("HR_comma_sep.csv")

In [ ]:
df

# Check Data imbalance

In [ ]:
import seaborn as sns

print(df["left"].value_counts())
df["left"].value_counts().plot(kind="bar")

# Convert categorical columns to label encoded

In [ ]:
le = LabelEncoder()
for column in df.columns:
    if df[column].dtype == object:
        df[column] = le.fit_transform(df[column])

In [ ]:
df

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()

df_scaled = std_scaler.fit_transform(df.drop("left", axis=1).to_numpy())
df_scaled = pd.DataFrame(df_scaled)
df_scaled

In [ ]:
X = df_scaled
y = df["left"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)
print(len(X_train))
print(len(X_val))
print(len(X_test))



# Split the data into training, testing, and validation sets



# Train a random forest model


In [ ]:
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(n_estimators=100, random_state=49, max_depth=4)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

# Plot calibration curve for raw model

In [ ]:
# Get the model probabilities on the test set (uncalibrated model)
probs_uncalibrated = model.predict_proba(X_test)[:, 1]

# Calculate Brier score for the uncalibrated model
brier_uncalibrated = brier_score_loss(y_test, probs_uncalibrated)
print(f"Brier Score for Uncalibrated Model: {round(brier_uncalibrated, 4)}")

# Compute the calibration curve for the uncalibrated model
(
    fraction_of_positives_uncalibrated,
    mean_predicted_value_uncalibrated,
) = calibration_curve(y_test, probs_uncalibrated, n_bins=5, strategy="uniform")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
plt.plot(
    mean_predicted_value_uncalibrated,
    fraction_of_positives_uncalibrated,
    label="Uncalibrated",
)
plt.xlabel("Mean predicted value")
plt.ylabel("Fraction of positives")
plt.title("Calibration curve (Test set)")
plt.legend()
plt.show()

# Calibration using Platt's scaling

In [ ]:
# Calibrate the model on the validation data using Platt's scaling
platt_scaling = CalibratedClassifierCV(model, method="sigmoid", cv="prefit")
platt_scaling.fit(X_val, y_val)

# Get the model probabilities on the test set (calibrated model - Platt's scaling)
probs_ps = platt_scaling.predict_proba(X_test)[:, 1]

# Calculate Brier score for the Platt's scaling calibrated model
brier_platt = brier_score_loss(y_test, probs_ps, pos_label=1)
print(f"Brier Score for Platt's Scaling Calibrated Model: {round(brier_platt, 4)}")

# Compute the calibration curve for Platt's scaling
fraction_of_positives_ps, mean_predicted_value_ps = calibration_curve(
    y_test, probs_ps, n_bins=5, strategy="uniform"
)

# Calibration using Platt's scaling

In [ ]:
# Calibrate the model on the validation data using isotonic regression
isotonic_regression = CalibratedClassifierCV(model, method="isotonic", cv="prefit")
isotonic_regression.fit(X_val, y_val)

# Get the model probabilities on the test set (calibrated model - Isotonic regression)
probs_ir = isotonic_regression.predict_proba(X_test)[:, 1]
# print(probs_ir)

# Calculate Brier score for the isotonic regression calibrated model
brier_isotonic = brier_score_loss(y_test, probs_ir)
print(
    f"Brier Score for Isotonic Regression Calibrated Model: {round(brier_isotonic, 4)}"
)
# print(X_test)

# Compute the calibration curve for isotonic regression
fraction_of_positives_ir, mean_predicted_value_ir = calibration_curve(
    y_test, probs_ir, n_bins=5, strategy="uniform"
)

# Create a calibration plot


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
plt.plot(
    mean_predicted_value_uncalibrated,
    fraction_of_positives_uncalibrated,
    label="Uncalibrated %s" % round(brier_uncalibrated, 4),
)
plt.plot(
    mean_predicted_value_ps,
    fraction_of_positives_ps,
    label="Platt's scaling %s" % round(brier_platt, 4),
    linestyle="-.",
)
plt.plot(
    mean_predicted_value_ir,
    fraction_of_positives_ir,
    label="Isotonic regression %s" % round(brier_isotonic, 4),
    linestyle="--",
)
plt.xlabel("Mean predicted value")
plt.ylabel("Fraction of positives")
plt.title("Calibration curves (Test set)")
plt.legend(loc="lower right")
plt.show()